# Introduction to LLM Prompting with Small Language Models

This notebook introduces how to work with **Small Language Models (SLMs)** running locally on your machine using the **llama-cpp-python** package. You will learn the fundamentals of LLM prompting — from a simple first query all the way to few-shot learning and streaming responses.

### What you will learn

1. How to load a local language model from a `.gguf` file
2. How to send prompts and receive responses
3. How to control response length and randomness
4. How to use system messages to shape model behavior
5. How to use few-shot examples to guide model output
6. How to stream responses token-by-token

### What is llama-cpp-python?

[llama-cpp-python](https://github.com/abetlen/llama-cpp-python) provides Python bindings for the [llama.cpp](https://github.com/ggml-org/llama.cpp) library — a high-performance C++ inference engine for running Large Language Models (LLMs) locally. It supports quantized GGUF models, which are compact enough to run efficiently on CPU.

### Key features

| Feature | Description |
|---------|-------------|
| **Local inference** | Runs entirely on your machine — no internet or API key needed |
| **GGUF model support** | Works with quantized models from Hugging Face |
| **Low memory footprint** | 4-bit, 8-bit, and other quantization levels supported |
| **OpenAI-compatible API** | Same message format as OpenAI's Chat API |
| **Cross-platform** | Works on Windows, macOS, and Linux |

### Resources

- llama-cpp-python [GitHub Repository](https://github.com/abetlen/llama-cpp-python)
- llama-cpp-python [Documentation](https://llama-cpp-python.readthedocs.io/)
- llama.cpp [GitHub Repository](https://github.com/ggml-org/llama.cpp)
- Hugging Face [GGUF Models](https://huggingface.co/models?search=gguf)

### Attribution

Notebook originally developed by Greg Merritt <[gmerritt@berkeley.edu](mailto:gmerritt@berkeley.edu)> and adapted by Eric Van Dusen.

---
## 1. Environment Setup

Before we can prompt a model, we need to:
1. Install the `llama-cpp-python` package
2. Locate the `.gguf` model file on disk
3. Load the model into memory

_This notebook assumes that at least one `.gguf` model file has already been downloaded. See `HuggingFace_Hub_Download_gguf.ipynb` for instructions on downloading a model._

### 1.1 Install llama-cpp-python

In [ ]:
# Install llama-cpp-python if it is not already available
try:
    from llama_cpp import Llama
except ImportError:
    %pip install llama-cpp-python
    from llama_cpp import Llama

### 1.2 Locate your model file

Your `.gguf` model file must be accessible on the local filesystem. Run the cells below that match your environment to confirm the file is there.

**Option A — Shared JupyterHub (e.g. Cal-ICOR workshop hub)**

In [ ]:
# List model files on the shared hub
!ls /home/jovyan/shared/

**Option B — Local machine**

In [ ]:
# List model files in a local directory (adjust path as needed)
!ls shared-rw

### 1.3 Set the model directory

Set `model_directory` to the folder that contains your `.gguf` file. Uncomment the line that matches your setup.

In [ ]:
# Shared JupyterHub
model_directory = "/home/jovyan/shared/"

# Local machine (uncomment and edit as needed)
# model_directory = "/Users/yourname/path/to/models"

### 1.4 Load the model

We load the model using the `Llama` class. The key parameters are:

| Parameter | Description |
|-----------|-------------|
| `model_path` | Full path to the `.gguf` model file |
| `n_ctx` | Context window size — how many tokens the model can "see" at once |
| `n_threads` | Number of CPU threads (`None` = auto-detect) |
| `verbose` | Print model loading details |
| `chat_format` | Chat template format (Qwen models use `"chatml"`) |

The model we use, `qwen2-1_5b-instruct-q4_0.gguf`, is a **1.5 billion-parameter Qwen2 model** that has been **quantized to 4-bit precision** to reduce its size and memory usage.

In [ ]:
import os

model_name = "qwen2-1_5b-instruct-q4_0.gguf"
model_path = os.path.join(model_directory, model_name)

model = Llama(
    model_path=model_path,
    n_ctx=2048,
    n_threads=None,
    verbose=True,
    chat_format="chatml"   # Qwen models use the ChatML template
)

print(f"\n✓ Model loaded: {model_name}")

---
## 2. Your First Prompt

We interact with the model through `create_chat_completion()`. This method takes a list of **messages** — each message has a `role` and `content`:

```python
messages = [
    {"role": "user",      "content": "Your question here"},
    {"role": "assistant", "content": "(previous model reply, if any)"},
]
```

The method returns a response object. The generated text is at:

```python
response["choices"][0]["message"]["content"]
```

Try editing `user_message` below to ask anything you like!

In [ ]:
user_message = "Who pays for tariffs on foreign manufactured goods — the consumer or the producer?"  # Try changing this!

messages = [
    {"role": "user", "content": user_message}
]

response = model.create_chat_completion(messages=messages)

print(response["choices"][0]["message"]["content"])

### Understanding the response object

The full response follows the OpenAI Chat Completions format:

```python
{
    "id": "chatcmpl-...",
    "object": "chat.completion",
    "choices": [
        {
            "message": {"role": "assistant", "content": "..."},
            "finish_reason": "stop"
        }
    ],
    "usage": {"prompt_tokens": 10, "completion_tokens": 50, "total_tokens": 60}
}
```

Run the cell below to inspect the full structure.

In [ ]:
# Inspect the full response object
import json
print(json.dumps(response, indent=2))

---
## 3. Controlling Generation Parameters

`create_chat_completion()` accepts several parameters that let you shape the model's output:

| Parameter | Description | Default |
|-----------|-------------|--------|
| `max_tokens` | Maximum number of tokens to generate | 16 |
| `temperature` | Randomness (0 = deterministic, higher = more varied) | 0.8 |
| `top_p` | Nucleus sampling threshold | 0.95 |
| `top_k` | Consider only the top-k most likely tokens | 40 |
| `repeat_penalty` | Penalise repeated tokens (1.0 = no penalty) | 1.1 |
| `stream` | Yield tokens one at a time instead of waiting for the full response | False |

### 3a. Limiting response length with `max_tokens`

Generation stops as soon as the token count reaches `max_tokens`, even mid-sentence.

In [ ]:
user_message = "What is the economic outcome of tariffs on foreign manufactured goods?"

response = model.create_chat_completion(
    messages=[{"role": "user", "content": user_message}],
    max_tokens=60   # Try smaller or larger values
)

print(response["choices"][0]["message"]["content"])

### 3b. Controlling creativity with `temperature`

At every step, the model computes a probability distribution over possible next tokens. **Temperature** rescales that distribution:

- **`temperature = 0`** — always pick the most likely token → identical outputs every run
- **`temperature ≈ 0.5`** — balanced creativity and coherence
- **`temperature = 1.0`** — high variety, occasionally less coherent
- **`temperature > 1.0`** — very random output

**First, let's run the same prompt three times at `temperature = 0` — outputs should be identical:**

In [ ]:
user_message = "How will tariffs affect the prices of foreign manufactured goods?"
temperature = 0.0   # Try changing this

for i in range(3):
    response = model.create_chat_completion(
        messages=[{"role": "user", "content": user_message}],
        max_tokens=30,
        temperature=temperature
    )
    print(f"Response {i + 1}: {response['choices'][0]['message']['content']}\n")

**Now increase `temperature` to `0.25` — responses start to diverge:**

In [ ]:
user_message = "How will tariffs affect elections?"
temperature = 0.25

for i in range(3):
    response = model.create_chat_completion(
        messages=[{"role": "user", "content": user_message}],
        max_tokens=30,
        temperature=temperature
    )
    print(f"Response {i + 1}: {response['choices'][0]['message']['content']}\n")

**At `temperature = 1.0`, responses are highly varied:**

In [ ]:
user_message = "How will tariffs affect elections?"
temperature = 1.0

for i in range(5):
    response = model.create_chat_completion(
        messages=[{"role": "user", "content": user_message}],
        max_tokens=30,
        temperature=temperature
    )
    print(f"Response {i + 1}: {response['choices'][0]['message']['content']}\n")

---
## 4. System Messages

A **system message** is a special instruction you give the model before the conversation begins. It sets the assistant's persona, tone, and constraints. The user never sees it — it's a hidden backstage direction.

```python
messages = [
    {"role": "system", "content": "Your instructions here..."},
    {"role": "user",   "content": "User's question"}
]
```

> **Note:** System messages are not guaranteed to stay secret — models can sometimes be coaxed into revealing them.

In [ ]:
system_message = """
You are a hard-working economics student at UC Berkeley.
You believe that memes, poems, and pop songs are the best way to communicate economic ideas.
Always answer in rap lyrics.
"""

user_message = "How will tariffs affect inflation?"

messages = [
    {"role": "system", "content": system_message},
    {"role": "user",   "content": user_message}
]

response = model.create_chat_completion(
    messages=messages,
    max_tokens=150
)

print(response["choices"][0]["message"]["content"])

Try editing `system_message` to give the model a different persona — a pirate economist, a five-year-old explaining the news, or a Shakespearean commentator.

---
## 5. Few-Shot Prompting

**Few-shot prompting** means giving the model example question–answer pairs *inside the prompt itself*. The model learns the expected response style from these examples and applies it to new questions.

We simply include alternating `user` / `assistant` messages before the real question:

```python
messages = [
    {"role": "system",    "content": "..."},
    {"role": "user",      "content": "Example question 1"},
    {"role": "assistant", "content": "Example answer 1"},
    {"role": "user",      "content": "Example question 2"},
    {"role": "assistant", "content": "Example answer 2"},
    {"role": "user",      "content": "Real question"}      # ← model answers this
]
```

The model doesn't "learn" in a training sense — it simply matches the pattern it sees in context.

### 5a. A few-shot example

We establish a style of concise, one-sentence economics explanations and then ask a new question.

In [ ]:
system_message = """
You are an economics tutor with a focus on international trade.
Answer concisely and clearly in one or two sentences.
"""

messages = [
    {"role": "system",    "content": system_message},
    # --- few-shot examples ---
    {"role": "user",      "content": "What is a tariff?"},
    {"role": "assistant", "content": "A tariff is a tax on imported goods, often used to protect domestic industries."},
    {"role": "user",      "content": "How do tariffs affect consumer prices?"},
    {"role": "assistant", "content": "Tariffs raise the cost of imported goods, which is usually passed on to consumers."},
    {"role": "user",      "content": "Can tariffs backfire?"},
    {"role": "assistant", "content": "Yes — they can trigger trade wars, hurt domestic exporters, and reduce overall economic efficiency."},
    # --- real question ---
    {"role": "user",      "content": "What is a real-world example of a tariff dispute?"}
]

response = model.create_chat_completion(
    messages=messages,
    max_tokens=200,
    temperature=0.7
)

print(response["choices"][0]["message"]["content"])

### 5b. How chat templating works under the hood

When you pass a messages list, llama-cpp-python converts it to the correct **chat template** for the model. For Qwen (ChatML format), your messages become:

```
<|im_start|>system
You are an economics tutor...
<|im_end|>
<|im_start|>user
What is a tariff?
<|im_end|>
<|im_start|>assistant
```

Using the wrong template causes the model to produce garbled output. The `chat_format="chatml"` argument we passed when loading ensures this is handled correctly.

### 5c. A note on "hallucinations"

You may hear that LLMs sometimes "hallucinate" — produce confident-sounding but factually wrong statements. This is a natural consequence of how these models work:

- An LLM predicts the *most statistically likely* next token, not the *most factually accurate* one.
- It has no external knowledge source at inference time.
- It cannot verify its own outputs.

This is not a "bug" — it is a fundamental property of next-token prediction. Understanding it helps you write better prompts and set appropriate expectations.

**Tip:** For factual tasks, add instructions like _"If you are unsure, say so"_ in your system message.

### 5d. Building a multi-turn conversation

To create a chatbot with memory, maintain a running list of messages and append each exchange:

```python
messages = [{"role": "system", "content": "You are a helpful assistant."}]

while True:
    user_input = input("You: ")
    if user_input.lower() == "quit":
        break

    messages.append({"role": "user", "content": user_input})

    response = model.create_chat_completion(messages=messages)
    reply = response["choices"][0]["message"]["content"]

    messages.append({"role": "assistant", "content": reply})
    print(f"Assistant: {reply}\n")
```

> **Key insight:** The LLM has no built-in memory. Your application re-sends the *entire conversation history* on every call. The model only "remembers" because you tell it everything each time.

---
## 6. Bonus: Streaming Responses

By default, `create_chat_completion()` waits until the model finishes generating before returning the full text. **Streaming** lets you see each token as it is produced — just like ChatGPT's typing effect.

Set `stream=True` to get a generator you can iterate over:

In [ ]:
user_message = "Explain the concept of comparative advantage in international trade."

stream = model.create_chat_completion(
    messages=[{"role": "user", "content": user_message}],
    max_tokens=150,
    stream=True
)

for chunk in stream:
    delta = chunk["choices"][0]["delta"]
    if "content" in delta:
        print(delta["content"], end="", flush=True)

print()  # newline at end

---
## Summary

In this notebook you learned how to:

| Topic | Key idea |
|-------|----------|
| **Load a model** | Use the `Llama` class with a `.gguf` file path |
| **Send a prompt** | Call `create_chat_completion()` with a `messages` list |
| **Limit length** | Use `max_tokens` to cap the response |
| **Control creativity** | Use `temperature` (0 = deterministic, higher = more varied) |
| **Set a persona** | Add a `{"role": "system", ...}` message at the start |
| **Guide output style** | Prepend example question/answer pairs (few-shot prompting) |
| **Stream tokens** | Set `stream=True` for real-time output |

### Next steps

- Try different `.gguf` models from Hugging Face (see `HuggingFace_Hub_Download_gguf.ipynb`)
- Experiment with different `temperature` and `top_p` values
- Build a simple interactive chatbot using the multi-turn pattern in section 5d
- Add a Gradio UI (see `Gradio_Chatbot_LlamaCpp.ipynb`)